In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

In [2]:
data = pd.read_parquet('../data/features/data_with_scores.parquet')

In [3]:
data.columns

Index(['period', 'Регистратор', 'contractor', 'КонтрагентИНН', 'account_dt',
       'СубконтоДт1', 'ВидСубконтоДт1', 'СубконтоДт2', 'ВидСубконтоДт2',
       'СубконтоДт3', 'ВидСубконтоДт3', 'account_kt', 'СубконтоКт1',
       'ВидСубконтоКт1', 'СубконтоКт2', 'ВидСубконтоКт2', 'СубконтоКт3',
       'ВидСубконтоКт3', 'dept_dt', 'dept_kt', 'amount', 'Содержание',
       'ТипДокумента', 'is_manual', 'hour', 'day_of_week', 'month',
       'is_weekend', 'is_night', 'is_mounth_end', 'is_quarter_end',
       'log_amount', 'is_negative_amount', 'abs_amount', 'account_pair',
       'pair_frequency', 'log_pair_frequency', 'is_rare_pair', 'pair_mean',
       'pair_std', 'amount_zscore', 'is_amount_outlier', 'contractor_freq',
       'log_contractor_frequency', 'contractor_ops_before',
       'is_first_operation', 'dept_dt_freq', 'dept_kt_freq', 'large_and_rare',
       'late_and_large', 'new_cont_and_large', 'manual_and_large',
       'first_contractor_pair', 'is_first_contractor_pair', 'date',
  

In [4]:
scaler = MinMaxScaler(feature_range=(0, 100))

ensemble_score = scaler.fit_transform(data[["ensemble_score"]]).flatten()

data['risk_score'] = ensemble_score

In [6]:
data['risk_score'].describe()

count    368014.000000
mean         12.802979
std           9.428386
min           0.000000
25%           5.189139
50%          10.713830
75%          17.948718
max         100.000000
Name: risk_score, dtype: float64

In [9]:
def explain_anomaly(row):
    reasons = []
    if row["is_rare_pair"]:
        reasons.append(f"Редкая пара счетов {row['account_dt']}→{row['account_kt']}")
    if row["is_amount_outlier"]:
        reasons.append(f"Сумма необычно большая для этого типа операции")
    if row["is_night"]:
        reasons.append(f"Операция в нерабочее время ({row['hour']}:00)")
    if row["is_weekend"]:
        reasons.append("Операция в выходной день")
    if row["is_first_operation"]:
        reasons.append(f"Первая операция с этим контрагентом")
    if row["is_manual"]:
        reasons.append("Ручная проводка")
    if row["is_negative_amount"]:
        reasons.append("Сторно (отрицательная сумма)")
    if not reasons:
        reasons.append("Комбинация факторов")
    return "; ".join(reasons)

data["explanation"] = data.apply(explain_anomaly, axis=1)

In [12]:
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment

report_data = data.nlargest(40, "risk_score")[[
    "period", "account_dt", "account_kt", "amount",
    "contractor", "Содержание", "ТипДокумента", "risk_score", "explanation"
]].copy()

report_data.columns = [
    "Дата", "Счет Дт", "Счет Кт", "Сумма",
    "Контрагент", "Содержание", "Тип документа", "Риск (0-100)", "Причина"
]

report_data["Риск (0-100)"] = report_data["Риск (0-100)"].round(1)

import os
os.makedirs("../reports", exist_ok=True)
report_data.to_excel("../reports/anomalies_for_review.xlsx", index=False, sheet_name="Аномалии")

wb = openpyxl.load_workbook("../reports/anomalies_for_review.xlsx")
ws = wb["Аномалии"]

red_fill = PatternFill(start_color="FFCCCC", end_color="FFCCCC", fill_type="solid")
yellow_fill = PatternFill(start_color="FFFFCC", end_color="FFFFCC", fill_type="solid")
header_fill = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")
header_font = Font(color="FFFFFF", bold=True)

for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal="center")

for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
    risk = row[7].value
    if risk and risk > 80:
        for cell in row:
            cell.fill = red_fill
    elif risk and risk > 60:
        for cell in row:
            cell.fill = yellow_fill

for col in ws.columns:
    max_length = max(len(str(cell.value or "")) for cell in col)
    ws.column_dimensions[col[0].column_letter].width = min(max_length + 2, 50)

from openpyxl.worksheet.datavalidation import DataValidation

# Добавляем пустые колонки для разметки
ws.cell(row=1, column=10, value="Оценка").fill = header_fill
ws.cell(row=1, column=10).font = header_font
ws.cell(row=1, column=11, value="Комментарий").fill = header_fill
ws.cell(row=1, column=11).font = header_font

# Выпадающий список для колонки "Оценка"
dv = DataValidation(
    type="list",
    formula1='"Ошибка,Норма,Непонятно"',
    allow_blank=True,
    showDropDown=False,  # False = стрелка выпадашки видна
)
ws.add_data_validation(dv)
dv.sqref = f"J2:J{ws.max_row}"

# Ширина новых колонок
ws.column_dimensions["J"].width = 15
ws.column_dimensions["K"].width = 40

wb.save("../reports/anomalies_for_review2.xlsx")
print("Отчёт сохранён: reports/anomalies_for_review2.xlsx")

Отчёт сохранён: reports/anomalies_for_review2.xlsx
